In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_data_dir():
    target = Path("Data/GEFCom2012_2level")
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / target
        if candidate.exists():
            return candidate
    return Path.cwd()

DATA_DIR = find_data_dir()
RAW_PATH = DATA_DIR / "Load_GEFCom2012.csv"
HOURLY_PATH = DATA_DIR / "Load_GEFCom2012_hourly.csv"
HIER_PATH = DATA_DIR / "hierarchy.csv"
SUM_MATRIX_PATH = DATA_DIR / "sum_matrix.csv"
HIER_INFO_PATH = DATA_DIR / "hierarchy_info.json"
NODE_VALUES_PATH = DATA_DIR / "node_values.npy"
NORM_PATH = DATA_DIR / "normalization_params.npy"
NORM_CSV_PATH = DATA_DIR / "node_values_normalized.csv"
LOG_OFFSET = 1.0
LOG_SKEW_THRESHOLD = 1.0
LOG_RATIO_THRESHOLD = 10.0
NORM_SKEW_THRESHOLD = 1.0
NORM_KURTOSIS_THRESHOLD = 5.0
TRAIN_RATIO = 0.8  # fit normalization stats on first 80% to avoid test leakage


In [ ]:
df_raw = pd.read_csv(RAW_PATH)
value_cols = [f"h{i}" for i in range(1, 25)]

required_cols = {"zone_id", "year", "month", "day"}
missing = required_cols - set(df_raw.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

df_long = df_raw.melt(
    id_vars=["zone_id", "year", "month", "day"],
    value_vars=value_cols,
    var_name="hour",
    value_name="load",
)

df_long["hour"] = df_long["hour"].str.replace("h", "", regex=False).astype(int)
df_long["timestamp"] = pd.to_datetime(df_long[["year", "month", "day"]]) + pd.to_timedelta(
    df_long["hour"] - 1, unit="h"
)

df_wide = (
    df_long.pivot(index="timestamp", columns="zone_id", values="load")
    .sort_index()
)

df_wide.columns = df_wide.columns.astype(int)

bottom_ids = list(range(1, 21))
missing_bottom = [z for z in bottom_ids if z not in df_wide.columns]
if missing_bottom:
    raise ValueError(f"Missing bottom zone_ids: {missing_bottom}")

df_wide[21] = df_wide[bottom_ids].sum(axis=1)
ordered_cols = [21] + bottom_ids

df_wide = df_wide[ordered_cols]
df_wide.index.name = "timestamp"

df_wide.to_csv(HOURLY_PATH)
print(f"Saved hourly data to {HOURLY_PATH}")
print(df_wide.head())
print(df_wide.shape)


In [ ]:
import csv
import json
from functools import lru_cache


def as_int_if_possible(value):
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        return text


children = {}
top_order = []
mid_order = []
leaf_order = []

with HIER_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        top = as_int_if_possible(row.get("Top"))
        mid = as_int_if_possible(row.get("Middle"))
        bottom = as_int_if_possible(row.get("Bottom"))
        path = [p for p in (top, mid, bottom) if p is not None]
        if not path:
            continue
        if top is not None and top not in top_order:
            top_order.append(top)
        if mid is not None and mid not in mid_order:
            mid_order.append(mid)
        leaf = path[-1]
        if leaf not in leaf_order:
            leaf_order.append(leaf)
        for parent, child in zip(path[:-1], path[1:]):
            children.setdefault(parent, [])
            if child not in children[parent]:
                children[parent].append(child)

bottom_order = leaf_order
node_order = top_order + mid_order + bottom_order
bottom_idx = {n: i for i, n in enumerate(bottom_order)}
index_map = {n: i for i, n in enumerate(node_order)}

@lru_cache(None)
def bottoms(node):
    if node in bottom_idx:
        return [node]
    res = []
    for ch in children.get(node, []):
        for b in bottoms(ch):
            if b not in res:
                res.append(b)
    return res

matrix = [[0] * len(bottom_order) for _ in node_order]
for i, node in enumerate(node_order):
    for b in bottoms(node):
        matrix[i][bottom_idx[b]] = 1

with SUM_MATRIX_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(matrix)

mid_to_bottom_indices = []
for mid in mid_order:
    idxs = []
    for b in bottoms(mid):
        idxs.append(index_map[b])
    mid_to_bottom_indices.append(idxs)

hierarchy_info = {
    "num_total_nodes": len(node_order),
    "num_bottom_nodes": len(bottom_order),
    "bottom_start_idx": len(top_order) + len(mid_order),
    "num_mid_nodes": len(mid_order),
    "num_top_nodes": len(top_order),
    "middle_levels": [],
    "middle_levels_provenance": {
        "generated_by": "DataProcessing.ipynb",
        "source": "hierarchy.csv",
        "validated_against": ["sum_matrix.csv", "mid_to_bottom_indices"],
    },
    "top_nodes": top_order,
    "mid_nodes": mid_order,
    "bottom_nodes": bottom_order,
    "node_order": node_order,
    "mid_to_bottom_indices": mid_to_bottom_indices,
}
with HIER_INFO_PATH.open("w", encoding="utf-8") as f:
    json.dump(hierarchy_info, f, ensure_ascii=True, indent=2)

print(f"Top: {top_order}")
print(f"Middle ({len(mid_order)}): {mid_order}")
print(f"Bottom ({len(bottom_order)}): {bottom_order}")
print(f"sum_matrix saved to {SUM_MATRIX_PATH} with shape ({len(node_order)}, {len(bottom_order)})")
print(f"hierarchy_info saved to {HIER_INFO_PATH}")


In [ ]:
def log_transform(df, log_offset=1.0):
    return np.log(df + log_offset)


def minmax_normalize(df):
    data = df.values
    min_val = float(data.min())
    max_val = float(data.max())
    denom = max_val - min_val if max_val != min_val else 1.0
    data_norm = (df - min_val) / denom
    params = {"method": "minmax", "min": min_val, "max": max_val}
    return data_norm, params


def zscore_normalize(df):
    data = df.values
    mean_val = float(data.mean())
    std_val = float(data.std())
    denom = std_val if std_val != 0 else 1.0
    data_norm = (df - mean_val) / denom
    params = {"method": "zscore", "mean": mean_val, "std": std_val}
    return data_norm, params


def should_log_transform(series, skew_threshold=1.0, ratio_threshold=10.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return False
    if values.min() < 0:
        return False
    positive = values[values > 0]
    if positive.size == 0:
        return False
    ratio = values.max() / positive.min()
    skew = pd.Series(values).skew()
    if skew is not None and skew > skew_threshold:
        return True
    return ratio > ratio_threshold


def choose_norm_method(series, skew_threshold=1.0, kurtosis_threshold=5.0):
    values = series.to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return "minmax", {"skew": None, "kurtosis": None}
    stats = pd.Series(values)
    skew = float(stats.skew())
    kurtosis = float(stats.kurtosis())
    if abs(skew) <= skew_threshold and abs(kurtosis) <= kurtosis_threshold:
        return "zscore", {"skew": skew, "kurtosis": kurtosis}
    return "minmax", {"skew": skew, "kurtosis": kurtosis}


def normalize_dataframe(df, log_offset=1.0, force_log=None, force_norm=None, train_ratio=TRAIN_RATIO):
    """Fit normalization on the first ``train_ratio`` fraction of rows only,
    then apply to the whole DataFrame.

    This prevents train/test leakage: the test period does not influence the
    log-transform decision, the norm-method decision, nor the fitted
    (mean/std) or (min/max) parameters.
    """
    T = len(df)
    train_T = max(1, int(T * train_ratio))
    df_train = df.iloc[:train_T]

    use_log = force_log if force_log is not None else should_log_transform(
        df_train.stack(),
        skew_threshold=LOG_SKEW_THRESHOLD,
        ratio_threshold=LOG_RATIO_THRESHOLD,
    )

    if use_log:
        df_base = log_transform(df, log_offset=log_offset)
        df_base_train = df_base.iloc[:train_T]
        data_space = "log"
    else:
        df_base = df.copy()
        df_base_train = df_train.copy()
        data_space = "raw"

    norm_method, stats = choose_norm_method(
        df_base_train.stack(),
        skew_threshold=NORM_SKEW_THRESHOLD,
        kurtosis_threshold=NORM_KURTOSIS_THRESHOLD,
    )
    if force_norm is not None:
        norm_method = force_norm

    # Fit on TRAIN, apply to FULL series
    if norm_method == "zscore":
        mean_val = float(df_base_train.values.mean())
        std_val = float(df_base_train.values.std())
        denom = std_val if std_val != 0 else 1.0
        data_norm = (df_base - mean_val) / denom
        norm_params = {"method": "zscore", "mean": mean_val, "std": std_val}
    else:
        min_val = float(df_base_train.values.min())
        max_val = float(df_base_train.values.max())
        denom = max_val - min_val if max_val != min_val else 1.0
        data_norm = (df_base - min_val) / denom
        norm_params = {"method": "minmax", "min": min_val, "max": max_val}

    params = {
        "use_log": bool(use_log),
        "log_offset": float(log_offset) if use_log else None,
        "data_space": data_space,
        "norm_method": norm_method,
        "decision_stats": stats,
        "train_ratio": float(train_ratio),
        "train_T": int(train_T),
        "total_T": int(T),
    }
    params.update(norm_params)

    values = data_norm.to_numpy(dtype=np.float32).reshape(-1, df.shape[1], 1)
    return values, params, use_log, norm_method

In [ ]:
df_hourly = pd.read_csv(HOURLY_PATH)
if "timestamp" in df_hourly.columns:
    df_hourly = df_hourly.set_index("timestamp")

col_map = {}
for c in df_hourly.columns:
    text = str(c)
    if text.isdigit():
        col_map[c] = int(text)
if col_map:
    df_hourly = df_hourly.rename(columns=col_map)

node_cols = node_order
missing_nodes = [n for n in node_cols if n not in df_hourly.columns]
if missing_nodes:
    raise ValueError(f"Missing nodes in hourly data: {missing_nodes}")

df_nodes = df_hourly[node_cols].astype(float)
values, norm_params, use_log, norm_method = normalize_dataframe(df_nodes, log_offset=LOG_OFFSET)

np.save(NODE_VALUES_PATH, values)
np.save(NORM_PATH, norm_params)
df_norm = pd.DataFrame(values.squeeze(-1), index=df_nodes.index, columns=df_nodes.columns)
df_norm.to_csv(NORM_CSV_PATH)
print(f"Saved normalized csv to {NORM_CSV_PATH}")
print("use_log:", use_log)
print("norm_method:", norm_method)
print("node_values shape:", values.shape)
print("normalization_params:", norm_params)
